In [2]:
# Cell 1: Setup
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Automatically adjust path if running inside the notebooks directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"✅ Success! Current Working Directory set to: {os.getcwd()}")

# Add the src folder to Python's search path
sys.path.append("src")
from utils import load_data, aqi_category

# Set clean visualization styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Cell 2: Load Processed Data
processed_path = "data/processed/clean_delhi.csv"

if os.path.exists(processed_path):
    print(f"📂 Loading pre-validated dataset from: {processed_path}")
    df = pd.read_csv(processed_path, parse_dates=["timestamp"])
else:
    print("⚠️ Clean processed file not found. Falling back to raw sample loader.")
    df = load_data()

print(f"Dataset summary: {len(df)} continuous hourly records loaded.")
print(f"Target cities for demo: {df['city'].unique().tolist()}")
df.head()

# Cell 3: AQI Distribution by City
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="city", y="aqi", ax=axes[0])
axes[0].set_title("AQI Distribution by City")

# Time series
for city in df["city"].unique():
    city_df = df[df["city"] == city]
    axes[1].plot(city_df["timestamp"], city_df["aqi"], label=city, alpha=0.7)
axes[1].set_title("AQI Over Time")
axes[1].legend()
plt.tight_layout()
plt.show()

# Cell 4: Model Comparison
with open("models_saved/final_comparison.json") as f:
    results = json.load(f)

comp_df = pd.DataFrame(results)
comp_df = comp_df.sort_values("MAE")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ["MAE", "RMSE", "MAPE"]
for ax, metric in zip(axes, metrics):
    colors = ["#2ecc71" if m == "Quantum Hybrid" else "#3498db" for m in comp_df["Model"]]
    bars = ax.barh(comp_df["Model"], comp_df[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(f"{metric} Comparison (Lower is Better)")
    # Add values
    for bar, val in zip(bars, comp_df[metric]):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va='center')
plt.tight_layout()
plt.show()

# Cell 5: Quantum Feature Selection Visualization
with open("models_saved/simple_quantum_metrics.json") as f:
    qm = json.load(f)

features = qm["features"]
print("Quantum-Selected Features:")
for i, f in enumerate(features, 1):
    print(f"  {i}. {f}")

# Feature importance chart
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = np.arange(len(features))
ax.barh(y_pos, [1]*len(features), color="#9b59b6")
ax.set_yticks(y_pos)
ax.set_yticklabels(features)
ax.set_xlabel("Selected (Quantum QAOA)")
ax.set_title("Features Selected by Quantum Algorithm")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Cell 6: Live Prediction Demo
import requests

# Start your API first: python src/api/main.py
BASE = "http://localhost:8000"

# Health check
r = requests.get(f"{BASE}/health")
print("API Status:", r.json())

# Forecast
forecast = {
    "city": "Delhi",
    "station_id": "station_0",
    "current_data": {
        "aqi": 180,
        "aqi_lag_1h": 175,
        "aqi_lag_24h": 190,
        "aqi_roll_mean_24h": 182,
        "aqi_roll_mean_168h": 178,
        "hour": 14,
        "humidity": 65,
        "wind_speed": 5,
        "temperature": 28
    },
    "horizons": [24, 48, 72],
    "model_type": "quantum_hybrid"
}

r = requests.post(f"{BASE}/forecast", json=forecast)
result = r.json()

print(f"\nForecast for {result['city']}:")
for f in result["forecasts"]:
    print(f"  {f['horizon_hours']}h: AQI {f['predicted_aqi']} ({f['category']})")

# Health Risk
risk = {
    "forecast_aqi": result["forecasts"][0]["predicted_aqi"],
    "user_profile": {
        "has_asthma": True,
        "elderly": False,
        "has_children": False,
        "outdoor_worker": True
    }
}

r = requests.post(f"{BASE}/health-risk", json=risk)
print(f"\nHealth Risk: {r.json()['risk_category']}")
print(f"Advisory: {r.json()['advisory']}")

SyntaxError: unexpected character after line continuation character (862696809.py, line 77)